# Multi-Task Data Collection

이 노트북은 라인 트레이싱 데이터와 손가락 분류 데이터를 하나의 UI에서 수집합니다.
**주의:** VSCode 환경 호환성을 위해 이미지 클릭 대신 슬라이더와 버튼을 이용해 라인 좌표를 수집합니다.

## ⚠️ [선택] 학습 데이터 초기화

처음부터 새로 데이터를 수집하려면 아래 셀을 실행한 뒤 **'🗑 초기화 실행'** 버튼을 누르세요.
- `dataset_multi/` 폴더 전체와 `best_model_multi.pth` 파일이 삭제됩니다.
- **실수로 누르지 않도록 2단계 확인(Confirm) 절차가 있습니다.**

In [1]:
import os
import shutil
import ipywidgets.widgets as _w
from IPython.display import display as _display

# ── 초기화 대상 정의 ──────────────────────────────────────────────────────────
_RESET_TARGETS = [
    ('dataset_multi', 'folder'),
    ('best_model_multi.pth', 'file'),
]

# ── 위젯 생성 ─────────────────────────────────────────────────────────────────
_reset_btn   = _w.Button(
    description='🗑 초기화 실행',
    button_style='danger',
    layout=_w.Layout(width='160px', height='36px')
)
_confirm_btn = _w.Button(
    description='✔ 확인 (삭제)',
    button_style='danger',
    layout=_w.Layout(width='140px', height='36px')
)
_cancel_btn  = _w.Button(
    description='✖ 취소',
    button_style='warning',
    layout=_w.Layout(width='100px', height='36px')
)
_status_lbl  = _w.HTML(value='')

# 처음엔 2단계 확인 패널 숨김
_confirm_pane = _w.HBox(
    [_w.HTML('<b style="color:red">정말 삭제하시겠습니까?</b>'),
     _confirm_btn, _cancel_btn],
    layout=_w.Layout(display='none')
)

def _on_reset_click(b):
    lines = []
    for name, kind in _RESET_TARGETS:
        icon  = '📁' if kind == 'folder' else '📄'
        state = '존재함' if os.path.exists(name) else '없음'
        lines.append(f'{icon} <b>{name}</b> — {state}')
    _status_lbl.value = (
        '<p style="color:#e65100">삭제 대상:<br>'
        + '<br>'.join(lines) + '</p>'
    )
    _confirm_pane.layout.display = ''

def _on_cancel_click(b):
    _confirm_pane.layout.display = 'none'
    _status_lbl.value = '<p style="color:gray">취소했습니다.</p>'

def _on_confirm_click(b):
    _confirm_pane.layout.display = 'none'
    log = []
    for name, kind in _RESET_TARGETS:
        if os.path.exists(name):
            if kind == 'folder':
                shutil.rmtree(name)
                log.append(f'✅ <b>{name}/</b> 폴더 삭제 완료')
            else:
                os.remove(name)
                log.append(f'✅ <b>{name}</b> 파일 삭제 완료')
        else:
            log.append(f'ℹ️ <b>{name}</b> — 존재하지 않아 건너뜀')
    _status_lbl.value = (
        '<p style="color:green">' + '<br>'.join(log)
        + '<br><b>초기화 완료!</b></p>'
    )

_reset_btn.on_click(_on_reset_click)
_confirm_btn.on_click(_on_confirm_click)
_cancel_btn.on_click(_on_cancel_click)

_display(_w.VBox([
    _w.HTML('<h4>📦 학습 데이터 초기화</h4>'),
    _reset_btn,
    _confirm_pane,
    _status_lbl,
]))

In [ ]:
import traitlets
import ipywidgets.widgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg

import os
import glob
from uuid import uuid1
import cv2
import numpy as np

DATASET_DIR = 'dataset_multi'
LINE_DIR = os.path.join(DATASET_DIR, 'line')
FINGER_DIR = os.path.join(DATASET_DIR, 'finger')
CLASSES = ['0', '1', '2', '3', '4', '5']

os.makedirs(LINE_DIR, exist_ok=True)
for c in CLASSES:
    os.makedirs(os.path.join(FINGER_DIR, c), exist_ok=True)
print('Directories created.')

Directories created.


: 

: 

: 

## 카메라 각도 세팅하기
Data를 수집할 때 사용한 이미지와 맞춰서 추론해야 하므로 카메라 각도를 고정합니다.

In [ ]:
from SCSCtrl import TTLServo
import time

def init_arm():
    print("로봇 팔을 초기 위치로 이동합니다...")
    
    # 서보모터 ID 1~5, 각도, 방향(1), 속도(150)
    # 3번 모터는 0도에서 기구적으로 걸리거나 가동 범위를 벗어날 수 있어 50으로 변경합니다.
    TTLServo.servoAngleCtrl(1, 0, 1, 150)
    TTLServo.servoAngleCtrl(2, 50, 1, 150)
    TTLServo.servoAngleCtrl(3, 50, 1, 150)  # 3번 모터 수정 (0 -> 50)
    TTLServo.servoAngleCtrl(4, 0, 1, 150)
    TTLServo.servoAngleCtrl(5, 50, 1, 150)

    time.sleep(1)
    print("초기화 완료!")

init_arm()

Succeeded to open the port
Succeeded to change the baudrate
로봇 팔을 초기 위치로 이동합니다...
초기화 완료!


In [ ]:
from SCSCtrl import TTLServo
import time

def init_arm():
    print("로봇 팔을 초기 위치로 이동합니다...")
    
    # 서보모터 ID 1~5, 각도, 방향(1), 속도(150)
    # 3번 모터는 0도에서 기구적으로 걸리거나 가동 범위를 벗어날 수 있어 50으로 변경합니다.
    TTLServo.servoAngleCtrl(1, 0, 1, 150)
    TTLServo.servoAngleCtrl(2, 50, 1, 150)
    TTLServo.servoAngleCtrl(3, 50, 1, 150)  # 3번 모터 수정 (0 -> 50)
    TTLServo.servoAngleCtrl(4, 0, 1, 150)
    TTLServo.servoAngleCtrl(5, -45, 1, 150)

    time.sleep(1)
    print("초기화 완료!")

init_arm()

로봇 팔을 초기 위치로 이동합니다...
초기화 완료!


In [5]:
import shutil

camera = Camera()

# 원본 카메라 이미지 위젯
image_widget = widgets.Image(format='jpeg', width=224, height=224)
traitlets.dlink((camera, 'value'), (image_widget, 'value'), transform=bgr8_to_jpeg)

# 라인 트레이싱용 타겟(미리보기) 위젯
target_widget = widgets.Image(format='jpeg', width=224, height=224)

mode_select = widgets.ToggleButtons(options=['Line Tracing', 'Finger Classification'], description='Mode:')

# Line variables
x_slider = widgets.FloatSlider(min=-1.0, max=1.0, step=0.001, description='x')
y_slider = widgets.FloatSlider(min=-1.0, max=1.0, step=0.001, description='y')
save_line_button = widgets.Button(description='Save Line Data', button_style='primary')

# Finger UI
class_select = widgets.ToggleButtons(options=CLASSES, description='Fingers:')
save_finger_button = widgets.Button(description='Save Finger', button_style='success')
count_label = widgets.HTML(value='')

# ── UI 내장 데이터 초기화 버튼 (2단계 확인) ──────────────────────────────────
reset_btn_ui     = widgets.Button(
    description='🗑 데이터 초기화',
    button_style='danger',
    layout=widgets.Layout(width='150px', height='34px')
)
reset_confirm_ui = widgets.Button(
    description='✔ 확인 (삭제)',
    button_style='danger',
    layout=widgets.Layout(width='130px', height='34px', display='none')
)
reset_cancel_ui  = widgets.Button(
    description='✖ 취소',
    button_style='warning',
    layout=widgets.Layout(width='90px', height='34px', display='none')
)
reset_status_ui  = widgets.HTML(value='')

def _show_confirm_ui(b):
    reset_confirm_ui.layout.display = ''
    reset_cancel_ui.layout.display  = ''
    reset_status_ui.value = (
        '<span style="color:red">'
        '⚠️ dataset_multi/ 전체와 best_model_multi.pth가 삭제됩니다!'
        '</span>'
    )

def _cancel_reset_ui(b):
    reset_confirm_ui.layout.display = 'none'
    reset_cancel_ui.layout.display  = 'none'
    reset_status_ui.value = '<span style="color:gray">취소했습니다.</span>'

def _do_reset_ui(b):
    reset_confirm_ui.layout.display = 'none'
    reset_cancel_ui.layout.display  = 'none'
    log = []
    if os.path.exists(DATASET_DIR):
        shutil.rmtree(DATASET_DIR)
        log.append('✅ dataset_multi/ 삭제')
    else:
        log.append('ℹ️ dataset_multi/ 없음')
    os.makedirs(LINE_DIR, exist_ok=True)
    for c in CLASSES:
        os.makedirs(os.path.join(FINGER_DIR, c), exist_ok=True)
    log.append('✅ 빈 dataset_multi/ 재생성')
    if os.path.exists('best_model_multi.pth'):
        os.remove('best_model_multi.pth')
        log.append('✅ best_model_multi.pth 삭제')
    else:
        log.append('ℹ️ best_model_multi.pth 없음')
    reset_status_ui.value = (
        '<span style="color:green">'
        + ' / '.join(log)
        + ' — 초기화 완료!</span>'
    )
    update_counts()

reset_btn_ui.on_click(_show_confirm_ui)
reset_confirm_ui.on_click(_do_reset_ui)
reset_cancel_ui.on_click(_cancel_reset_ui)

reset_box = widgets.VBox([
    widgets.HTML('<hr><b>🗑 학습 데이터 초기화</b>'),
    widgets.HBox([reset_btn_ui, reset_confirm_ui, reset_cancel_ui]),
    reset_status_ui,
])
# ─────────────────────────────────────────────────────────────────────────────

def update_counts():
    counts = {}
    for c in CLASSES:
        counts[c] = len(glob.glob(os.path.join(FINGER_DIR, c, '*.jpg')))
    line_count = len(glob.glob(os.path.join(LINE_DIR, '*.jpg')))
    
    html = f'<h4>Counts:</h4><b>Line:</b> {line_count}<br>'
    for c in CLASSES:
        html += f'<b>Finger {c}:</b> {counts[c]}<br>'
    count_label.value = html

def save_finger_snapshot(change):
    c = class_select.value
    filepath = os.path.join(FINGER_DIR, c, f"{c}_{str(uuid1())}.jpg")
    with open(filepath, 'wb') as f:
        f.write(image_widget.value)
    update_counts()
save_finger_button.on_click(save_finger_snapshot)

def save_line_snapshot(change):
    x, y = x_slider.value, y_slider.value
    filepath = os.path.join(LINE_DIR, f"xy_{int(x*112+112):03d}_{int(y*112+112):03d}_{str(uuid1())}.jpg")
    with open(filepath, 'wb') as f:
        f.write(image_widget.value)
    update_counts()
save_line_button.on_click(save_line_snapshot)

# 실시간 미리보기 그리기
def display_xy(camera_image):
    image = np.copy(camera_image)
    x = x_slider.value
    y = y_slider.value
    x_px = int(x * 112 + 112)
    y_px = int(y * 112 + 112)
    image = cv2.circle(image, (x_px, y_px), 8, (0, 255, 0), 3)
    image = cv2.circle(image, (112, 224), 8, (0, 0, 255), 3)
    image = cv2.line(image, (x_px, y_px), (112, 224), (255, 0, 0), 3)
    return bgr8_to_jpeg(image)

traitlets.dlink((camera, 'value'), (target_widget, 'value'), transform=display_xy)

# 슬라이더 값이 바뀔 때 target_widget을 강제 업데이트하도록 추가 연결
def on_slider_change(change):
    target_widget.value = display_xy(camera.value)
x_slider.observe(on_slider_change, names='value')
y_slider.observe(on_slider_change, names='value')

update_counts()

finger_box = widgets.VBox([class_select, save_finger_button])
line_box = widgets.VBox([x_slider, y_slider, save_line_button, widgets.HTML("<b>슬라이더로 X, Y를 맞추고 'Save Line Data' 버튼을 누르세요.</b>")])

def on_mode_change(change):
    if change['new'] == 'Line Tracing':
        finger_box.layout.display = 'none'
        line_box.layout.display = 'block'
        image_widget.layout.display = 'none'
        target_widget.layout.display = 'block'
    else:
        finger_box.layout.display = 'block'
        line_box.layout.display = 'none'
        image_widget.layout.display = 'block'
        target_widget.layout.display = 'none'

mode_select.observe(on_mode_change, names='value')
on_mode_change({'new': mode_select.value})

display(widgets.VBox([mode_select, widgets.HBox([image_widget, target_widget]), finger_box, line_box, count_label, reset_box]))

In [10]:
camera.stop()